# nb_00 — (instructor only) regenerate the source data

You normally **don't run this**. The lab consumes pre-generated files hosted in
the repo's `/data` folder. This reproduces them deterministically if you want to
change volumes or reshape the scenario. Run it in a plain Python kernel; it writes
a local `./data` folder you then commit to your fork.

In [ ]:
"""
Meridian People Analytics lab — synthetic source data generator.
Deterministic. Produces the external files learners download & host:

  reference/workers.csv              initial worker snapshot   (SCD2 load 1)
  reference/workers_delta.csv        later worker snapshot     (SCD2 load 2)
  reference/cost_centers.csv         cost-centre master        (SCD1, carries RLS region)
  feeds/pay_bands_feed.json          nested pay-grid history   (SCD2 from history)
  feeds/fx_rates.csv                 monthly FX -> CAD
  events/workforce_events_YYYY-MM.csv  60 monthly HR event extracts (pipeline ForEach)

Design hooks used by the modules:
  - dedup:      ~4% of events replayed with same event_id, later ingest_ts
  - quarantine: negative amounts, orphan employee_id, bad dates, unknown currency
  - conform:    two source systems emit iso2 vs iso3 work-country codes
  - SCD2:       pay grids re-benchmarked annually; worker records change; as-of joins
"""
import json, os, random
from datetime import date, datetime, timedelta
import numpy as np
import pandas as pd

SEED = 20260904
rng = np.random.default_rng(SEED)
random.seed(SEED)

ROOT = os.path.dirname(os.path.abspath(__file__))
DATA = os.path.join(ROOT, "data")
for sub in ["reference", "feeds", "events"]:
    d = os.path.join(DATA, sub)
    os.makedirs(d, exist_ok=True)
    for f in os.listdir(d):
        os.remove(os.path.join(d, f))

START = date(2021, 1, 1)
END   = date(2025, 12, 31)
MONTHS = pd.period_range(START, END, freq="M")
YEARS  = [2021, 2022, 2023, 2024, 2025]
DELTA_DATE = date(2023, 7, 1)          # second worker snapshot

# ------------------------------------------------------------------ pay grids
GROUPS = {   # group -> (level-1 midpoint CAD, per-level step CAD)
    "CR": (50000,  6000),   # Clerical & Regulatory
    "AS": (55000,  7000),   # Administrative Services
    "PA": (60000,  9000),   # Program & Administrative
    "FI": (70000, 11000),   # Financial Management
    "EC": (72000, 12000),   # Economics & Social Science
    "IT": (78000, 13000),   # Information Technology
    "EN": (80000, 13000),   # Engineering
    "EX": (130000,20000),   # Executive
}
LEVELS = [1, 2, 3, 4, 5]
# annual economic increase applied each Jan (2023 has a catch-up)
ECON = {2021: 0.000, 2022: 0.023, 2023: 0.045, 2024: 0.028, 2025: 0.022}

def band_grid():
    """returns grid[(group,level,year)] = (min,mid,max) and the JSON feed"""
    grid = {}
    classifications = []
    for g, (base, step) in GROUPS.items():
        for lv in LEVELS:
            mid0 = base + (lv - 1) * step
            hist = []
            factor = 1.0
            for y in YEARS:
                factor *= (1 + ECON[y])
                mid = round(mid0 * factor, 0)
                lo  = round(mid * 0.88, 0)
                hi  = round(mid * 1.12, 0)
                grid[(g, lv, y)] = (lo, mid, hi)
                hist.append({"effective_date": f"{y}-01-01",
                             "band_min": lo, "band_mid": mid, "band_max": hi})
            classifications.append({"group": g, "level": lv, "band_history": hist})
    feed = {
        "source": "Meridian Compensation & Classification",
        "as_of": END.isoformat(),
        "currency": "CAD",
        "methodology": "Pay grid re-benchmarked each fiscal year via collective agreement",
        "classifications": classifications,
    }
    with open(os.path.join(DATA, "feeds", "pay_bands_feed.json"), "w") as f:
        json.dump(feed, f, indent=2)
    return grid

def mid_asof(grid, g, lv, d):
    return grid[(g, lv, d.year)][1]
def band_asof(grid, g, lv, d):
    return grid[(g, lv, d.year)]

# ------------------------------------------------------------------ FX
# work countries for international offices (rest are Canada / CAD)
INTL = [   # iso2, iso3, currency
    ("US","USA","USD"),("GB","GBR","GBP"),("FR","FRA","EUR"),("DE","DEU","EUR"),
    ("JP","JPN","JPY"),("AU","AUS","AUD"),("SG","SGP","SGD"),("BR","BRA","BRL"),
    ("AE","ARE","AED"),("KE","KEN","KES"),("IN","IND","INR"),("MX","MEX","MXN"),
]
CAN = ("CA","CAN","CAD")
FX_ANCHOR = {"CAD":1.0,"USD":1.35,"GBP":1.70,"EUR":1.45,"JPY":0.0091,"AUD":0.89,
             "SGD":1.00,"BRL":0.26,"AED":0.37,"KES":0.0090,"INR":0.016,"MXN":0.075}
def build_fx():
    rows = []
    for ccy in FX_ANCHOR:
        anchor = FX_ANCHOR[ccy]; rate = anchor
        for m in MONTHS:
            if ccy == "CAD":
                r = 1.0
            else:
                rate *= float(rng.normal(1.0, 0.02))
                r = round(max(rate, anchor*0.3), 8)
            rows.append({"rate_month": m.strftime("%Y-%m"), "currency": ccy,
                         "cad_per_unit": r})
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(DATA, "feeds", "fx_rates.csv"), index=False)
    return {(x.rate_month, x.currency): x.cad_per_unit for x in df.itertuples()}

# ------------------------------------------------------------------ cost centres
REGIONS = ["National Capital Region","Atlantic","Quebec","Ontario",
           "Prairies","Pacific","International"]
BRANCHES = ["Operations","Corporate Services","Digital & Technology","Policy & Programs",
            "Finance & Administration","Legal Services","Communications",
            "Audit & Evaluation","International Affairs"]
BUSINESS_LINES = ["Service Delivery","Enabling Services","Oversight","Strategy"]
def build_cost_centers(n=120):
    rows = []
    for i in range(1, n+1):
        region = str(rng.choice(REGIONS, p=[.30,.08,.14,.16,.10,.10,.12]))
        rows.append({
            "cost_center_id": f"CC{i:04d}",
            "cost_center_name": f"{random.choice(BRANCHES)} — Unit {i:03d}",
            "branch": str(rng.choice(BRANCHES)),
            "hr_region": region,
            "business_line": str(rng.choice(BUSINESS_LINES)),
        })
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(DATA, "reference", "cost_centers.csv"), index=False)
    return df

# ------------------------------------------------------------------ workers
FIRST = ["Avery","Jordan","Riley","Casey","Morgan","Taylor","Cameron","Skyler",
         "Drew","Quinn","Reese","Harper","Emerson","Rowan","Sasha","Devon",
         "Noor","Amara","Kai","Leah","Mateo","Priya","Owen","Zoe","Hassan","Mei"]
LAST  = ["Tremblay","Roy","Gagnon","Lee","Chen","Singh","Patel","Nguyen","Brown",
         "Wilson","MacDonald","Cote","Bouchard","Kaur","Ali","Diallo","Okafor",
         "Ferreira","Novak","Ivanov","Yamamoto","Rossi","Dubois","Martin"]
ETYPES = ["Indeterminate","Term","Casual","Student"]
DIRECTORATES = ["Service Delivery","Platform Engineering","Grants & Contributions",
                "Financial Reporting","Workforce Programs","Regulatory Affairs",
                "Data & Analytics","Field Operations","Client Relations",
                "Infrastructure","Security & Compliance","Program Design"]
def draw_group_level():
    g = str(rng.choice(list(GROUPS.keys()),
             p=[.10,.12,.24,.10,.10,.16,.10,.08]))
    if g == "EX":
        lv = int(rng.choice([1,2,3], p=[.6,.3,.1]))
    else:
        lv = int(rng.choice(LEVELS, p=[.28,.28,.22,.14,.08]))
    return g, lv

def assign_country(region):
    if region == "International":
        iso2, iso3, ccy = INTL[int(rng.integers(0, len(INTL)))]
        return iso2, iso3, ccy
    return CAN

def build_workers(cost_centers, n=5000):
    cc = cost_centers.to_dict("records")
    rows = []
    for i in range(1, n+1):
        g, lv = draw_group_level()
        rows.append({
            "employee_id": f"EMP{i:05d}",
            "full_name": f"{random.choice(FIRST)} {random.choice(LAST)}",
            "classification_group": g,
            "classification_level": lv,
            "directorate": str(rng.choice(DIRECTORATES)),
            "employment_type": str(rng.choice(ETYPES, p=[.72,.18,.06,.04])),
            "home_cost_center_id": str(rng.choice([c["cost_center_id"] for c in cc])),
            "snapshot_date": START.isoformat(),
        })
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(DATA, "reference", "workers.csv"), index=False)

    # second snapshot (2023-07-01): ~350 changed + 200 brand-new employees
    delta = df.copy()
    delta["snapshot_date"] = DELTA_DATE.isoformat()
    change_idx = rng.choice(df.index, size=350, replace=False)
    changed = set()
    for idx in change_idx:
        roll = rng.random()
        if roll < 0.45 and delta.at[idx,"classification_level"] < 5 \
           and delta.at[idx,"classification_group"] != "EX":       # promotion
            delta.at[idx,"classification_level"] += 1
        elif roll < 0.75:                                          # deployment
            delta.at[idx,"directorate"] = str(rng.choice(DIRECTORATES))
        else:                                                      # conversion
            order = ["Student","Casual","Term","Indeterminate"]
            cur = delta.at[idx,"employment_type"]
            delta.at[idx,"employment_type"] = order[min(len(order)-1, order.index(cur)+1)] \
                if cur in order else cur
        changed.add(df.at[idx,"employee_id"])
    new_rows = []
    for j in range(1, 201):
        g, lv = draw_group_level()
        new_rows.append({
            "employee_id": f"EMP{n+j:05d}",
            "full_name": f"{random.choice(FIRST)} {random.choice(LAST)}",
            "classification_group": g, "classification_level": lv,
            "directorate": str(rng.choice(DIRECTORATES)),
            "employment_type": str(rng.choice(ETYPES, p=[.5,.25,.15,.10])),
            "home_cost_center_id": str(rng.choice([c["cost_center_id"] for c in cc])),
            "snapshot_date": DELTA_DATE.isoformat(),
        })
    delta = pd.concat([delta, pd.DataFrame(new_rows)], ignore_index=True)
    delta.to_csv(os.path.join(DATA, "reference", "workers_delta.csv"), index=False)

    # build an in-memory 'as-of state' per employee for event generation
    state = {}
    v1 = df.set_index("employee_id")
    v2 = delta[delta["snapshot_date"]==DELTA_DATE.isoformat()].set_index("employee_id")
    cc_region = cost_centers.set_index("cost_center_id")["hr_region"].to_dict()
    for eid in v1.index:
        r1 = v1.loc[eid]
        rec = {
            "exists_from": START,
            "v1": (r1.classification_group, int(r1.classification_level),
                   r1.home_cost_center_id),
            "v2": None,
        }
        if eid in changed and eid in v2.index:
            r2 = v2.loc[eid]
            rec["v2"] = (r2.classification_group, int(r2.classification_level),
                         r2.home_cost_center_id)
        state[eid] = rec
    for eid in v2.index:                 # brand-new employees
        if eid not in state:
            r2 = v2.loc[eid]
            state[eid] = {"exists_from": DELTA_DATE,
                          "v1": None,
                          "v2": (r2.classification_group, int(r2.classification_level),
                                 r2.home_cost_center_id)}
    return df, state, cc_region

# ------------------------------------------------------------------ events
EVENT_TYPES = ["Step Increment","Performance Pay","Lateral Deployment",
               "Promotion","Leave Start","Leave Return","Hire","Departure"]
COMP_SETTING = {"Hire","Promotion","Step Increment"}   # set a new base salary
def state_asof(rec, d):
    if d < rec["exists_from"]:
        return None
    if d >= DELTA_DATE and rec["v2"] is not None:
        return rec["v2"]
    return rec["v1"] if rec["v1"] is not None else rec["v2"]

def build_events(grid, fx, state, cc_region, per_month=2000):
    iso3_to_iso2 = {c[1]: c[0] for c in INTL} | {CAN[1]: CAN[0]}
    ccy_by_iso3  = {c[1]: c[2] for c in INTL} | {CAN[1]: CAN[2]}
    # precompute region->intl country pool for international cost centres
    emp_ids = list(state.keys())
    all_rows = []
    eid = 0
    for m in MONTHS:
        mstart = m.start_time.date()
        n = per_month + int(rng.integers(-150, 150))
        # candidate employees existing this month
        picks = rng.choice(emp_ids, size=n*2, replace=True)
        made = 0
        for e in picks:
            if made >= n:
                break
            rec = state[e]
            st = state_asof(rec, mstart)
            if st is None:
                continue
            g, lv, cc = st
            region = cc_region.get(cc, "National Capital Region")
            # work country + currency from region
            if region == "International":
                iso2, iso3, ccy = INTL[int(rng.integers(0, len(INTL)))]
            else:
                iso2, iso3, ccy = CAN
            etype = str(rng.choice(EVENT_TYPES,
                        p=[.34,.14,.10,.05,.13,.12,.06,.06]))
            day = mstart + timedelta(days=int(rng.integers(0, 27)))
            src = "CORE_HR" if rng.random() < 0.6 else "PAYROLL"
            wcc = iso3 if src == "CORE_HR" else iso2      # conform split
            # amount
            if etype in COMP_SETTING:
                lo, mid, hi = band_asof(grid, g, lv, day)
                placement = rng.random()
                if placement < 0.10:      base_cad = lo * float(rng.uniform(0.82,0.99))
                elif placement > 0.95:    base_cad = hi * float(rng.uniform(1.01,1.15))
                else:                     base_cad = mid * float(rng.uniform(0.90,1.10))
                fxr = fx.get((day.strftime("%Y-%m"), ccy), FX_ANCHOR.get(ccy,1.0))
                amount_local = round(base_cad / (fxr or 1.0), 2)
            elif etype == "Performance Pay":
                lo, mid, hi = band_asof(grid, g, lv, day)
                fxr = fx.get((day.strftime("%Y-%m"), ccy), FX_ANCHOR.get(ccy,1.0))
                amount_local = round(mid * float(rng.uniform(0.04,0.14)) / (fxr or 1.0), 2)
            else:
                amount_local = ""     # non-comp events carry no amount
            all_rows.append([f"EVT{eid:08d}", day.isoformat(), e, cc, g, lv, etype,
                             amount_local, ccy, wcc,
                             (datetime.combine(day, datetime.min.time())
                              + timedelta(hours=int(rng.integers(6,40)))).isoformat(sep=" "),
                             src])
            eid += 1
            made += 1

    cols = ["event_id","event_date","employee_id","cost_center_id",
            "classification_group","classification_level","event_type",
            "amount_local","local_currency","work_country_code","ingest_ts","source_system"]
    df = pd.DataFrame(all_rows, columns=cols)

    # duplicates (~4%): same event_id, later ingest_ts
    dup = df.sample(frac=0.04, random_state=SEED).copy()
    dup["ingest_ts"] = (pd.to_datetime(dup["ingest_ts"])
                        + pd.to_timedelta(rng.integers(1,72,len(dup)), unit="h")
                       ).dt.strftime("%Y-%m-%d %H:%M:%S")
    df = pd.concat([df, dup], ignore_index=True)

    # dirty rows (~1.5%): negatives, orphan employee, bad date, unknown currency
    dirty = list(df.sample(frac=0.015, random_state=SEED+1).index)
    rng.shuffle(dirty)
    q = len(dirty)//4
    # negatives only on comp/bonus rows (rows with a numeric amount)
    numeric_mask = df["amount_local"].astype(str).str.match(r"^-?\d+(\.\d+)?$")
    neg_pool = [i for i in dirty[:q] if numeric_mask.get(i, False)]
    for i in neg_pool:
        df.at[i,"amount_local"] = -abs(float(df.at[i,"amount_local"]))
    df.loc[dirty[q:2*q], "employee_id"] = "EMP99999"          # orphan
    df.loc[dirty[2*q:3*q], "event_date"] = "1900-01-01"       # bad date
    df.loc[dirty[3*q:], "local_currency"] = "XXX"             # unknown ccy

    # partition into monthly files by INGEST month (always valid)
    df["_m"] = pd.to_datetime(df["ingest_ts"], errors="coerce").dt.to_period("M")
    df["_m"] = df["_m"].fillna(pd.Period("2021-01","M"))
    df.loc[df["_m"] < pd.Period("2021-01","M"), "_m"] = pd.Period("2021-01","M")
    df.loc[df["_m"] > pd.Period("2025-12","M"), "_m"] = pd.Period("2025-12","M")
    for mm, gdf in df.groupby("_m"):
        gdf = gdf.drop(columns="_m").sample(frac=1, random_state=SEED)
        gdf.to_csv(os.path.join(DATA,"events",f"workforce_events_{mm}.csv"), index=False)
    return df.drop(columns="_m")

# ------------------------------------------------------------------ run
print("pay grids ..."); grid = band_grid()
print("fx ...");        fx   = build_fx()
print("cost centres ..."); cost_centers = build_cost_centers()
print("workers ...");   workers, state, cc_region = build_workers(cost_centers)
print("events ...");    events = build_events(grid, fx, state, cc_region)

# ------------------------------------------------------------------ validation
print("\n--- VALIDATION ---")
clean = events[(events.employee_id!="EMP99999") & (events.local_currency!="XXX")
               & (events.event_date!="1900-01-01")]
print(f"total event rows (incl dupes+dirty): {len(events):,}")
print(f"distinct event_id                  : {events.event_id.nunique():,}")
print(f"dupe rows                          : {len(events)-events.event_id.nunique():,}")
print(f"orphan employee rows               : {(events.employee_id=='EMP99999').sum():,}")
print(f"unknown-currency rows              : {(events.local_currency=='XXX').sum():,}")
amt = pd.to_numeric(events.amount_local, errors="coerce")
print(f"negative-amount rows               : {(amt<0).sum():,}")
print(f"bad-date rows                      : {(events.event_date=='1900-01-01').sum():,}")
print(f"iso2/iso3 split (PAYROLL/CORE_HR)  : "
      f"{(events.source_system=='PAYROLL').sum():,} / {(events.source_system=='CORE_HR').sum():,}")
print("event type mix:")
print(events.event_type.value_counts().to_string())

# RI
valid_emp = set(workers.employee_id) | set(pd.read_csv(os.path.join(DATA,'reference','workers_delta.csv')).employee_id)
assert clean.employee_id.isin(valid_emp).all(), "orphan employee leak"
months = set(k[0] for k in fx)
for c in [x for x in clean.local_currency.unique()]:
    for mm in [p.strftime("%Y-%m") for p in MONTHS]:
        assert (mm,c) in fx, f"missing fx {mm} {c}"
nver = 8*5*5
print(f"pay-band versioned rows            : {nver} (8 groups x 5 levels x 5 years)")

# compa-ratio sanity: comp events near band, drift positive over time
comp = clean[clean.event_type.isin(list(COMP_SETTING))].copy()
print(f"comp-setting events                : {len(comp):,}")
nfiles = len(os.listdir(os.path.join(DATA,"events")))
print(f"monthly event files                : {nfiles}")
print("done.")
